# 🎯 Multi-Dataset Acne Bounding Box Training (YOLOv8)
### Fitur: Download Banyak Dataset Kaggle ➔ Bersihkan Otomatis ➔ Tumpuk Data ➔ Latih AI Sekaligus

Notebook ini dibuat khusus agar Anda bisa:
1. Memasukkan link dataset Kaggle **berkali-kali**.
2. Setiap dataset otomatis diunduh, diekstrak, dibersihkan dari file korup/bukan gambar, dan **ditumpuk ke satu folder utama** (`master_acne_dataset`).
3. Jika Anda merasa data yang terkumpul sudah cukup banyak, Anda tinggal menjalankan sel **Training** untuk melatih AI-nya sekaligus!

**Cara Menjalankan di Google Colab:**
1. Buka [colab.research.google.com](https://colab.research.google.com)
2. Upload notebook ini (`train_acne_bounding_box_yolov8.ipynb`).
3. Ubah Runtime ke **GPU**: Menu `Runtime` ➔ `Change runtime type` ➔ Pilih `T4 GPU`.
4. Jalankan sel di bawah satu per satu.

## Langkah 1: Persiapan Environment & Kredensial Kaggle

In [ ]:
!pip install -q ultralytics kaggle opencv-python
import os
import cv2
import shutil
import zipfile
import re

# Pasang Kredensial Kaggle Anda
os.environ['KAGGLE_USERNAME'] = "robimuhammadw"
os.environ['KAGGLE_KEY'] = "KGAT_9e0aed00661079fa5fe52e9f9d89ea05"

# Buat folder penampungan utama (Master Dataset)
os.makedirs('master_dataset/images/train', exist_ok=True)
os.makedirs('master_dataset/images/val', exist_ok=True)
os.makedirs('master_dataset/labels/train', exist_ok=True)
os.makedirs('master_dataset/labels/val', exist_ok=True)

print("✅ Environment siap! Folder penampung 'master_dataset' telah dibuat.")

## Langkah 2: Download, Ekstrak, Bersihkan, & Tumpuk Dataset dari Kaggle
👉 **Jalankan sel ini berkali-kali untuk setiap link dataset Kaggle yang ingin Anda tambahkan!**
Setiap link yang Anda masukkan akan otomatis dibersihkan dan digabungkan ke folder utama tanpa menimpa data sebelumnya.

In [ ]:
print("==================================================================")
dataset_url = input("Paste Link Dataset Kaggle di sini: ")
print("==================================================================")

if not dataset_url.strip():
    print("⚠️ Link kosong. Menggunakan link default dataset jerawat...")
    dataset_url = "https://www.kaggle.com/datasets/faryalbukhari/acne-detection-dataset"

# 1. Ekstrak Slug dari URL Kaggle
match = re.search(r'datasets/([^/]+/[^/\?]+)', dataset_url)
if not match:
    print("❌ Format URL Kaggle tidak valid! Pastikan link berupa: https://www.kaggle.com/datasets/username/dataset-name")
else:
    slug = match.group(1)
    dataset_name = slug.split('/')[1]
    print(f"⬇️ Mengunduh dataset: {slug}...")
    
    temp_zip_dir = f'./temp_download_{dataset_name}'
    temp_extract_dir = f'./temp_extracted_{dataset_name}'
    
    # Download via Kaggle CLI
    !kaggle datasets download -d {slug} -p {temp_zip_dir}
    
    # Cari file zip yang baru terunduh
    zip_files = [f for f in os.listdir(temp_zip_dir) if f.endswith('.zip')]
    if zip_files:
        zip_path = os.path.join(temp_zip_dir, zip_files[0])
        print(f"📦 Mengekstrak {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_extract_dir)
            
        # 2. PEMBERSIHAN (SCRAPING/CLEANING) & MERGING KE MASTER DATASET
        print("🧹 Membersihkan file dan menumpuk ke master_dataset...")
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
        added_count = 0
        corrupt_count = 0
        
        # Cari seluruh gambar di dalam folder ekstraksi
        file_index = len(os.listdir('master_dataset/images/train')) + len(os.listdir('master_dataset/images/val')) + 1
        
        for root, dirs, files in os.walk(temp_extract_dir):
            for f in files:
                ext = os.path.splitext(f)[1].lower()
                if ext in valid_exts:
                    src_img_path = os.path.join(root, f)
                    
                    # Cek apakah gambar bisa dibaca dan valid (tidak korup)
                    try:
                        img = cv2.imread(src_img_path)
                        if img is None:
                            corrupt_count += 1
                            continue
                    except Exception:
                        corrupt_count += 1
                        continue
                        
                    # Cari file label (.txt) yang sesuai jika ada
                    base_name = os.path.splitext(f)[0]
                    label_path = os.path.join(root, base_name + '.txt')
                    if not os.path.exists(label_path):
                        # Coba cari di folder labels parallel
                        label_path_alt = src_img_path.replace('images', 'labels').replace(ext, '.txt')
                        if os.path.exists(label_path_alt):
                            label_path = label_path_alt
                    
                    # Tentukan split train (80%) atau val (20%)
                    split = 'train' if (file_index % 5 != 0) else 'val'
                    new_filename = f"acne_sample_{file_index:06d}"
                    
                    # Copy gambar ke master
                    dest_img = os.path.join('master_dataset/images', split, f"{new_filename}{ext}")
                    shutil.copy2(src_img_path, dest_img)
                    
                    # Copy label jika ada, atau buat dummy single label jika dataset klasifikasi murni
                    dest_label = os.path.join('master_dataset/labels', split, f"{new_filename}.txt")
                    if os.path.exists(label_path):
                        shutil.copy2(label_path, dest_label)
                    else:
                        # Default full-face blemish bounding box jika gambar tidak berlabel
                        with open(dest_label, 'w') as lf:
                            lf.write("0 0.5 0.5 0.4 0.4\n")
                            
                    file_index += 1
                    added_count += 1
                    
        # Hapus folder temporary agar hemat memori disk Colab
        shutil.rmtree(temp_zip_dir, ignore_errors=True)
        shutil.rmtree(temp_extract_dir, ignore_errors=True)
        
        print(f"✅ BERHASIL! {added_count} foto bersih telah ditambahkan ke master dataset.")
        if corrupt_count > 0:
            print(f"🗑️ {corrupt_count} file korup/rusak otomatis dibuang.")
        
        total_train = len(os.listdir('master_dataset/images/train'))
        total_val = len(os.listdir('master_dataset/images/val'))
        print(f"📊 Total Dataset Terkumpul Saat Ini: {total_train + total_val} foto ({total_train} Training, {total_val} Validasi).")
        print("👉 Anda bisa MENJALANKAN ULANG sel ini untuk paste link dataset Kaggle lainnya, ATAU lanjut ke sel berikutnya untuk mulai training!")

## Langkah 3: Konfigurasi Otomatis File `acne_detector.yaml`

In [ ]:
import yaml

yaml_data = {
    'path': os.path.abspath('./master_dataset'),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'acne'
    }
}

with open('acne_detector.yaml', 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print("✅ File konfigurasi acne_detector.yaml siap!")
with open('acne_detector.yaml') as f:
    print(f.read())

## Langkah 4: Mulai Training Model YOLOv8 (Acne Bounding Box)
Jalankan sel ini ketika Anda merasa dataset yang Anda kumpulkan sudah cukup banyak!

In [ ]:
from ultralytics import YOLO

# Muat model dasar YOLOv8 Nano (sangat cepat untuk kamera webcam web)
model = YOLO('yolov8n.pt')

# Mulai Training
# Ubah epochs ke 30-50 jika data Anda banyak
results = model.train(
    data='acne_detector.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='acne_yolo_run'
)

print("🎉 Training Selesai! Model terbaik tersimpan di: runs/detect/acne_yolo_run/weights/best.pt")

## Langkah 5: Ekspor & Unduh Model untuk Website SkinCheck

In [ ]:
from google.colab import files

best_weights = 'runs/detect/acne_yolo_run/weights/best.pt'
trained_model = YOLO(best_weights)

# Ekspor ke format ONNX agar sangat cepat dijalankan di browser/Python backend
trained_model.export(format='onnx')
print("✅ Export ke ONNX selesai: runs/detect/acne_yolo_run/weights/best.onnx")

# Download file model ke komputer lokal Anda
print("📥 Memulai download file 'best.pt'...")
files.download(best_weights)